# NB-5: Scaling Benchmark — Memory Capacity vs Performance

Tests CSAM memory system performance (latency, throughput, retrieval quality) at
increasing NPC/memory counts. Runs locally — no Groq API calls required for the
core scaling test (uses `--no-llm` flag for pure memory/retrieval benchmarking).

**Runs:**
- Scaling sweep: NPCs × memories × queries at multiple sizes
- Optionally with LLM QA (requires GROQ_API_KEY) for end-to-end scaling

**Output directory:** `results/nb5_scaling/`  
**Time estimate:** ~5–15 min (no LLM), ~30–60 min (with LLM)

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — (Optional) Set API key for LLM-enabled scaling
Skip this step if running with `USE_LLM = False` in Step 3.

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

key = _load_secret('GROQ_API_KEY')
if key:
    os.environ['GROQ_API_KEY'] = key
    with open('.env', 'w') as f:
        f.write(f'GROQ_API_KEY={key}\n')
    print('GROQ_API_KEY loaded (LLM-enabled scaling available)')
else:
    print('No GROQ_API_KEY — will run with --no-llm (word-overlap scoring only)')

## Step 3 — Configure

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
MAX_NPCS  = 50    # number of NPC agents to test (50 for fast, 100+ for paper)
MEMORIES  = 100   # memories per NPC
QUERIES   = 10    # queries per NPC
USE_LLM   = bool(os.environ.get('GROQ_API_KEY'))  # auto-detected from Step 2
MODEL     = 'llama-3.1-8b-instant'
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR = os.path.join(REPO_DIR, 'results', 'nb5_scaling')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Output dir: {OUT_DIR}')
print(f'Max NPCs:   {MAX_NPCS}')
print(f'Memories:   {MEMORIES} per NPC')
print(f'Queries:    {QUERIES} per NPC')
print(f'Use LLM:    {USE_LLM} (model: {MODEL if USE_LLM else "N/A"})')

## Step 4 — Run scaling benchmark
Tests HNSW retrieval latency and throughput as memory store grows.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

out_path = os.path.join(OUT_DIR, f'scaling_npcs{MAX_NPCS}_mem{MEMORIES}_q{QUERIES}.json')

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_scaling',
    '--max-npcs', str(MAX_NPCS),
    '--memories', str(MEMORIES),
    '--queries',  str(QUERIES),
    '--output',   out_path,
]
if not USE_LLM:
    cmd.append('--no-llm')
    print('Running in --no-llm mode (word-overlap scoring, no API calls)')
else:
    cmd += ['--model', MODEL]
    print(f'Running with LLM ({MODEL})')

print(f'\nCMD: {" ".join(cmd[2:])}\n')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Scaling benchmark complete')
print(f'Output: {out_path}')

## Step 5 — Run larger-scale sweep (multiple NPC counts)
Tests at 10, 25, 50, 100 NPCs to show O(log n) retrieval scaling.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

NPC_COUNTS = [10, 25, 50, 100]
sweep_outputs = []

for npc_count in NPC_COUNTS:
    out_path = os.path.join(OUT_DIR, f'scaling_sweep_npcs{npc_count}.json')
    if os.path.exists(out_path):
        print(f'[SKIP] npcs={npc_count} already exists')
        sweep_outputs.append(out_path)
        continue

    cmd = [
        sys.executable, '-m', 'csam_project.benchmarks.benchmark_scaling',
        '--max-npcs', str(npc_count),
        '--memories', str(MEMORIES),
        '--queries',  str(QUERIES),
        '--output',   out_path,
        '--no-llm',  # sweep always no-llm to avoid API costs
    ]
    print(f'Running sweep: npcs={npc_count}...')
    result = subprocess.run(cmd, capture_output=False, text=True)
    status = 'OK' if result.returncode == 0 else 'FAIL'
    print(f'[{status}] npcs={npc_count}')
    if result.returncode == 0:
        sweep_outputs.append(out_path)

print(f'\nSweep complete: {len(sweep_outputs)}/{len(NPC_COUNTS)} runs succeeded')

## Step 6 — Results summary

In [ ]:
import json, os, glob

result_files = sorted(glob.glob(os.path.join(OUT_DIR, '*.json')))

print('=' * 70)
print('SCALING BENCHMARK RESULTS')
print('=' * 70)
print(f'{"File":<45} {"NPCs":>6} {"Avg Lat":>10} {"F1":>8}')
print('-' * 70)

for fp in result_files:
    try:
        with open(fp) as f: d = json.load(f)
        name     = os.path.basename(fp)[:44]
        npcs     = d.get('total_npcs', d.get('num_npcs', '?'))
        lat      = d.get('avg_retrieval_ms', d.get('avg_latency_ms', 0))
        f1       = d.get('avg_f1', d.get('overall_f1', 0))
        print(f'{name:<45} {str(npcs):>6} {lat:>10.2f} {f1:>8.4f}')
    except Exception as e:
        print(f'{os.path.basename(fp):<45} ERROR: {e}')

print(f'\nTotal result files: {len(result_files)}')

## Step 7 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb5_scaling'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')